<a href="https://colab.research.google.com/github/BumRushBangz/Summer26Projects/blob/main/Ch5FromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np

df = pd.read_csv("sample_data/california_housing_train.csv")
y = df['median_house_value'].tolist()
# We are trying to predict house value using k-fold cv (cross validation).

# Pick any predictors here. Add a column name to this list:
predictors = ['median_income', 'housing_median_age', 'total_rooms']
predictor_cols = [df[p].tolist() for p in predictors]
# One list per predictor (p).

# Build the design matrix (1 + predictors):
X_multi = []
for i in range(len(y)):
    row = [1] # This is the intercept
    for col in predictor_cols:
        row.append(col[i])  # For each predictor, add its value for row i.
    X_multi.append(row)
    # They now look like: [1, income_i, age_i, rooms_i].
X_multi = np.array(X_multi) # Matrix multiplication needs array.

y = np.array(y)
import random
random.seed(0)
indices = list(range(n))
random.shuffle(indices)
X_multi = X_multi[indices]
y = y[indices]

def fit_ols(X, y):
    X = np.array(X)
    y = np.array(y)
    return np.linalg.inv(X.T @ X) @ X.T @ y

In [34]:
k = 10
# Any value of k up until k=n will suffice, but be computationally more
# difficult, and risk overfitting.
n = len(y)
fold_size = n // k
# fold_size = n // k drops leftover rows when n isn't divisible by k.
# e.g. n=17, k=5 gives fold_size=3, so only 15 rows used, and last 2 ignored.
# sklearn's KFold instead spreads the remainder out: the first (n % k) folds
# get one extra row each, so every row gets used and folds are as even as
# possible. To do that by hand I'd give each fold its own size (base or base+1)
# and let "start" go forward from the previous fold's "end" instead of
# jumping by a fixed fold_size.

avg_errors = []
k_values = [2,3,4,5,6,7,8,9,10]
for k in k_values:

  fold_size = n // k
  MSEs = []

  for fold in range(k):
      start = fold * fold_size
      end = start + fold_size
      X_test = X_multi[start:end]
      y_test = y[start:end]
      # Now we're gonna have to combine everything before and everything after the
      # test fold:
      X_train = np.concatenate([X_multi[:start], X_multi[end:]])
      y_train = np.concatenate([y[:start], y[end:]])
      beta = fit_ols(X_train, y_train)  # Fit on this fold's train,
                                        # return coefficients.
      y_pred = X_test @ beta            # Predict this fold.
      mse = np.mean((y_test - y_pred)**2) # Find this fold's error.
      MSEs.append(mse)                  # Add it to the empty error list.

  cv_mse = np.mean(MSEs)
  # Find the average MSE across all folds.

  avg_error = cv_mse**0.5
  print ("On average, the predicted house value is off by" , round(avg_error,2),
       "dollars when we train on", k, "folds.")
  avg_errors.append((k, avg_error))
  # Double paranthesis so they're stored as a pair.

def error(pair): # Pairs were stored as (k, error).
    return pair[1] # Give back just the second term.

best_k = min(avg_errors, key=error)
# To find the "best" k, we take minimum error value.

print ("The best k value to train on is", best_k[0],
       "which gives us an average inaccuracy of", round(best_k[1],2),
       'dollars.')

On average, the predicted house value is off by 80519.64 dollars when we train on 2 folds.
On average, the predicted house value is off by 80515.09 dollars when we train on 3 folds.
On average, the predicted house value is off by 80515.68 dollars when we train on 4 folds.
On average, the predicted house value is off by 80509.58 dollars when we train on 5 folds.
On average, the predicted house value is off by 80522.67 dollars when we train on 6 folds.
On average, the predicted house value is off by 80520.73 dollars when we train on 7 folds.
On average, the predicted house value is off by 80515.26 dollars when we train on 8 folds.
On average, the predicted house value is off by 80515.85 dollars when we train on 9 folds.
On average, the predicted house value is off by 80512.25 dollars when we train on 10 folds.
The best k value to train on is 5 which gives us an average inaccuracy of 80509.58 dollars.
